In [3]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

# ============================================
# 1. СРЕДА
# ============================================
env = gym.make("FrozenLake-v1", is_slippery=False)

n_states = env.observation_space.n
n_actions = env.action_space.n

# ============================================
# 2. ONE-HOT кодирование состояния
# ============================================
def one_hot(state):
    vec = np.zeros(n_states)
    vec[state] = 1
    return vec

# ============================================
# 3. Q-сеть
# ============================================
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.net(x)

policy_net = DQN()
target_net = DQN()
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)

# ============================================
# 4. Replay buffer
# ============================================
memory = deque(maxlen=10000)

def store_transition(s, a, r, s_next, done):
    memory.append((s, a, r, s_next, done))

def sample_batch(batch_size):
    batch = random.sample(memory, batch_size)
    s, a, r, s_next, d = zip(*batch)
    return (
        torch.tensor(s, dtype=torch.float32),
        torch.tensor(a),
        torch.tensor(r, dtype=torch.float32),
        torch.tensor(s_next, dtype=torch.float32),
        torch.tensor(d, dtype=torch.float32),
    )

# ============================================
# 5. Параметры
# ============================================
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05
batch_size = 64

# ============================================
# 6. ОБУЧЕНИЕ
# ============================================
episodes = 1000

for ep in range(episodes):
    state, _ = env.reset()
    state = one_hot(state)

    done = False
    total_reward = 0

    while not done:
        # epsilon-greedy
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                q_vals = policy_net(torch.tensor(state))
                action = torch.argmax(q_vals).item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state_oh = one_hot(next_state)

        store_transition(state, action, reward, next_state_oh, done)

        state = next_state_oh
        total_reward += reward

        # обучение
        if len(memory) > batch_size:
            s, a, r, s_next, d = sample_batch(batch_size)

            q_values = policy_net(s)
            q_value = q_values.gather(1, a.unsqueeze(1)).squeeze()

            with torch.no_grad():
                next_q = target_net(s_next).max(1)[0]
                target = r + gamma * next_q * (1 - d)

            loss = nn.MSELoss()(q_value, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # обновление target сети
    if ep % 50 == 0:
        target_net.load_state_dict(policy_net.state_dict())

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if ep % 100 == 0:
        print(f"Episode {ep}, reward {total_reward}, epsilon {epsilon:.3f}")

# ============================================
# 7. ТЕСТ (визуализация)
# ============================================
state, _ = env.reset()
done = False

print("\nAgent learned policy:\n")

while not done:
    print(env.render())
    state_tensor = torch.tensor(one_hot(state), dtype=torch.float32)

    with torch.no_grad():
        action = torch.argmax(policy_net(state_tensor)).item()

    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

print(env.render())
print("Finished!")

Episode 0, reward 0, epsilon 0.995


C:\Users\nick\AppData\Local\Temp\ipykernel_15672\1892228109.py:58: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  torch.tensor(s, dtype=torch.float32),


RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float